In [0]:
dbutils.secrets.get(scope="magic_", key="cloudpwd")

In [0]:
%sql
CREATE CONNECTION IF NOT EXISTS gcp
TYPE mysql
OPTIONS (
  host '34.123.166.158',
  port '3306',
  user 'devuser',
  password secret('magic_', 'cloudpwd')
 --{{secrets/my-scope/mysql-password}}
);

In [0]:
%sql
CREATE FOREIGN CATALOG IF NOT EXISTS gcp_mysql_fc_wd36
USING CONNECTION gcp;

In [0]:
%sql
DESCRIBE connection gcp;

In [0]:
%sql
select * from gcp_mysql_fc_wd36.logistics.shipments1;

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS prodcatalog_wd36.logistics_wd36.bronze_shipments1
USING DELTA
AS
SELECT *
FROM gcp_mysql_fc_wd36.logistics.shipments1
""")

In [0]:
row_count = spark.sql("""
select * 
from gcp_mysql_fc_wd36.logistics.shipments1
where updated_at > (
    SELECT COALESCE(MAX(updated_at), '1970-01-01') 
    FROM prodcatalog_wd36.logistics_wd36.bronze_shipments1
)
""").count()

print(row_count)

dbutils.jobs.taskValues.set(key="row_count", value=row_count)